In [ ]:
from cv2 import imread,cvtColor,COLOR_BGR2RGB,createMergeMertens,imwrite,NORM_MINMAX,normalize,COLOR_RGB2BGR,resize, INTER_AREA
from numpy import clip, power, random as np_random, float32, uint8, save, load, array
from random import uniform
from matplotlib import pyplot as plt
from transformers import ViTFeatureExtractor, ViTModel
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from torch import no_grad,cat
from time import time, perf_counter
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
from glob import glob
import os
import torch.nn.functional as F
from timm import create_model
from torchvision import transforms


base_path = "/Users/priyanshusharma912o/ML_project/dataset" #project path
orig_dir = os.path.join(base_path, "original") #original images path
enh_dir = os.path.join(base_path, "enhanced") #enhanced images path

os.makedirs(enh_dir, exist_ok=True) #create enhanced images directory (if not present)
for cls in ["cats", "dogs"]:
    os.makedirs(os.path.join(enh_dir, cls), exist_ok=True) #create enhanced images directory (if not present)

def multi_exposure_enhance(image_rgb, num_exposures, bin_size): #function to enhance images
    """
    image_rgb: expects RGB image in uint8 or float range [0,1]
    returns: binned image in float32 with values in [0,1], RGB order
    """
    # ensure float32 and normalized to [0,1]
    img = image_rgb.astype(float32) / 255.0
    synthetic_exposures = [] #list to store synthetic exposures

    for _ in range(num_exposures):
        # STEP 3: Reduced variability for better similarity
        exposure_factor = uniform(-1, 1)  
        gamma = uniform(0.9, 1.1)

        exposed = clip(img * exposure_factor, 0.0, 1.0) 
        exposed = power(exposed, gamma)

        # STEP 3: Reduced noise injection
        noise = np_random.normal(0, 0.001, img.shape).astype(float32)
        exposed = clip(exposed + noise, 0.0, 1.0)

        # Mertens works with float32 images (0..1), append as float32
        synthetic_exposures.append(exposed.astype(float32))

    merge_mertens = createMergeMertens() # an object of Mertens class
    fused = merge_mertens.process(synthetic_exposures)  # float32, usually in [0,1] or small range

    # ensure fused is float32 and in [0,1]
    fused = clip(fused.astype(float32), 0.0, 1.0)

    # pixel binning (average pooling)
    h, w, c = fused.shape
    h_new, w_new = h // bin_size, w // bin_size
    fused_cropped = fused[:h_new * bin_size, :w_new * bin_size]# avoids border artifacts
    binned = fused_cropped.reshape(h_new, bin_size, w_new, bin_size, c).mean(axis=(1, 3)) # avoids using for loop

    return binned  # still RGB, float32, range ~[0,1]


/Users/priyanshusharma912o/miniconda3/envs/ML_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
for cls in ["cats", "dogs"]:
    orig_folder = os.path.join(orig_dir, cls) #folder for original cats and dogs
    enh_folder = os.path.join(enh_dir, cls) #folder for enhanced cats and dogs
    image_paths = [os.path.join(orig_folder, f) for f in os.listdir(orig_folder)] #list of image paths of the target folder

    for img_path in tqdm(image_paths, desc=f"Processing {cls}"):
        try:
            # Read (BGR), convert to RGB for internal processing
            bgr = imread(img_path)
            if bgr is None:
                continue
            rgb = cvtColor(bgr, COLOR_BGR2RGB)

            # Enhance (returns RGB float in [0,1])
            # STEP 3: Reduced num_exposures from 12 to 5
            #binned_image = multi_exposure_enhance(rgb, num_exposures=5, bin_size=2)
            #binned_image = multi_exposure_enhance1(rgb, num_exposures=5, bin_size=2)
            binned_image = multi_exposure_enhance(rgb, num_exposures=12, bin_size=2)
            #binned_image = simple_downsample(rgb, bin_size=2)

            # Normalize to 0..255 and convert to uint8
            enhanced_norm = normalize(binned_image, None, 0, 255, NORM_MINMAX)
            enhanced_uint8 = enhanced_norm.astype(uint8)

            # Convert back to BGR for saving with OpenCV
            enhanced_bgr = cvtColor(enhanced_uint8, COLOR_RGB2BGR)

            # Save
            filename = os.path.basename(img_path)
            save_path = os.path.join(enh_folder, filename)
            imwrite(save_path, enhanced_bgr)

        except Exception as e:
            print(f"Error processing {img_path}: {e}")


Processing dogs: 100%|██████████| 12270/12270 [27:34<00:00,  7.41it/s]


In [ ]:
# 1. Base setup 
feat_dir  = os.path.join(base_path, "features")
os.makedirs(feat_dir, exist_ok=True) # make folder for storing feature vectors

# 1. Load ViT model with dynamic image size support
model = create_model("vit_base_patch16_224", pretrained=True, dynamic_img_size=True)#allows variable image sizes(important for loading the preprocessed image)
model.eval().to('cpu')

# We'll take the CLS token embedding as feature
def get_vit_features(img_tensor):
    with no_grad():
        # Use the model's forward_features which handles variable sizes
        features = model.forward_features(img_tensor)
        # Return CLS token (first token)
        return features[:, 0]  # CLS embedding (768-D)

# 2. Transform — STEP 2: Use ImageNet normalization
to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet mean
                         std=[0.229, 0.224, 0.225]),    # ImageNet std
])

# 3. Feature extraction function
def extract_vit_feature(image_path, model):
    try:
        img = Image.open(image_path).convert("RGB")
        
        # Get original size
        orig_w, orig_h = img.size
        
        # Make dimensions divisible by 16 (patch size)
        # Round up to nearest multiple of 16
        new_h = ((orig_h + 15) // 16) * 16
        new_w = ((orig_w + 15) // 16) * 16
        
        # Resize to make divisible by 16 (minimal resize, preserves aspect ratio info)
        if new_h != orig_h or new_w != orig_w:
            img = img.resize((new_w, new_h), Image.BILINEAR)
        
        # Convert to tensor - NO forced 224x224!
        img_tensor = to_tensor(img).unsqueeze(0).to('cpu')

        # Calculate number of patches (tokens) based on actual image size
        h, w = img_tensor.shape[-2], img_tensor.shape[-1]
        num_patches_h, num_patches_w = h // 16, w // 16
        total_tokens = num_patches_h * num_patches_w + 1  # +1 for CLS token

        start = time()
        feat = get_vit_features(img_tensor)
        end = time()

        feature = feat.squeeze().cpu().numpy()
        return feature, (end - start), total_tokens

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None, 0.0, 0

# 4. Folder-wise processing
def process_split(split):
    src_root = os.path.join(base_path, split)
    dst_root = os.path.join(feat_dir, split)
    os.makedirs(dst_root, exist_ok=True)

    total_images, total_time, total_tokens = 0, 0.0, 0

    for cls in ["cats", "dogs"]:
        src_folder = os.path.join(src_root, cls)
        dst_folder = os.path.join(dst_root, cls)
        os.makedirs(dst_folder, exist_ok=True)

        image_files = [f for f in os.listdir(src_folder)
                       if f.lower().endswith((".jpg", ".png", ".jpeg"))]

        for fname in tqdm(image_files, desc=f"Extracting {split}/{cls}"):
            fpath = os.path.join(src_folder, fname)
            out_path = os.path.join(dst_folder,
                                    os.path.splitext(fname)[0] + ".npy")
            if os.path.exists(out_path):
                continue

            feat, t, tokens = extract_vit_feature(fpath, model)
            if feat is not None:
                save(out_path, feat)
                total_images += 1
                total_time += t
                total_tokens += tokens

    return total_images, total_time, total_tokens

# 5. Run for both datasets
print("Starting feature extraction (timm ViT)...\n")

num_orig, time_orig, tokens_orig = process_split("original")
num_enh,  time_enh, tokens_enh  = process_split("enhanced")

# 6. Summary with comparison table
print("\n" + "="*70)
print("FEATURE EXTRACTION SUMMARY - COMPARISON")
print("="*70)
print(f"{'Metric':<35} {'Original':<17} {'Enhanced':<17}")
print("-"*70)
print(f"{'Images processed':<35} {num_orig:<17} {num_enh:<17}")
print(f"{'Total tokens':<35} {tokens_orig:<17,} {tokens_enh:<17,}")
print(f"{'Average tokens per image':<35} {tokens_orig/num_orig if num_orig > 0 else 0:<17.1f} {tokens_enh/num_enh if num_enh > 0 else 0:<17.1f}")
print(f"{'Total time (seconds)':<35} {time_orig:<17.2f} {time_enh:<17.2f}")
print(f"{'Average time per image (seconds)':<35} {time_orig/num_orig if num_orig > 0 else 0:<17.4f} {time_enh/num_enh if num_enh > 0 else 0:<17.4f}")
print("-"*70)

# Calculate improvement metrics
if num_orig > 0 and num_enh > 0:
    token_reduction = (1 - (tokens_enh/num_enh) / (tokens_orig/num_orig)) * 100
    speedup = (time_orig/num_orig) / (time_enh/num_enh)
    print(f"Token reduction: {token_reduction:.1f}%")
    print(f"Speedup factor: {speedup:.2f}x")

print("="*70)


Starting feature extraction (timm ViT)...



Extracting original/dogs:   0%|          | 21/12269 [00:06<50:19,  4.06it/s]  /Users/priyanshusharma912o/miniconda3/envs/ML_project/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Extracting enhanced/dogs: 100%|██████████| 12269/12269 [16:39<00:00, 12.28it/s]


FEATURE EXTRACTION SUMMARY - COMPARISON
Metric                              Original          Enhanced         
----------------------------------------------------------------------
Images processed                    24559             24559            
Total tokens                        15,188,929        3,896,163        
Average tokens per image            618.5             158.6            
Total time (seconds)                7316.09           2017.58          
Average time per image (seconds)    0.2979            0.0822           
----------------------------------------------------------------------
Token reduction: 74.3%
Speedup factor: 3.63x


In [ ]:
orig_feats = []
enh_feats = []

for animal in ["cats", "dogs"]:
    orig_paths = glob(os.path.join(feat_dir, "original", animal, "*.npy"))
    enh_paths  = glob(os.path.join(feat_dir, "enhanced", animal, "*.npy"))
    for f in orig_paths:
        orig_feats.append(load(f))
    for f in enh_paths:
        enh_feats.append(load(f))

orig_feats = array(orig_feats)  # shape (N, 768)
enh_feats = array(enh_feats)    # shape (N, 768)
print(f"Loaded {orig_feats.shape[0]} original and {enh_feats.shape[0]} enhanced features.")
print(f"Original features shape: {orig_feats.shape}")
print(f"Enhanced features shape: {enh_feats.shape}")

#compute cosine similarity for corresponding images
def stem(path): 
    return os.path.splitext(os.path.basename(path))[0]

# Build lookup dictionaries
orig_dict, enh_dict = {}, {}
for animal in ["cats", "dogs"]:
    for path in glob(os.path.join(feat_dir, "original", animal, "*.npy")):
        orig_dict[stem(path)] = load(path)
    for path in glob(os.path.join(feat_dir, "enhanced", animal, "*.npy")):
        enh_dict[stem(path)] = load(path)

# Compute cosine similarities for each animal category
print("\n" + "="*50)
print("COSINE SIMILARITY: ORIGINAL vs ENHANCED")
print("="*50)

for animal in ["cats", "dogs"]:
    similarities = []
    for key, orig_feat in orig_dict.items():
        enh_feat = enh_dict.get(key)
        if enh_feat is None:
            continue
        # Compute cosine similarity between corresponding original and enhanced features
        sim = cosine_similarity([orig_feat], [enh_feat])[0][0]
        similarities.append(sim)
    
    avg_sim = sum(similarities) / len(similarities) if similarities else 0
    print(f"{animal.capitalize():<10} Average similarity: {avg_sim:.4f} ({len(similarities)} pairs)")

print("="*50)


Loaded 24559 original and 24760 enhanced features.
Original features shape: (24559, 768)
Enhanced features shape: (24760, 768)

COSINE SIMILARITY: ORIGINAL vs ENHANCED
Cats       Average similarity: 0.7544 (12300 pairs)
Dogs       Average similarity: 0.7544 (12300 pairs)


In [5]:
feat_dir  = os.path.join(base_path, "features")

In [ ]:
results = {}

# --- Train models separately for each set ---
for set_name in ["original", "enhanced"]:
    print(f"\nProcessing feature set: {set_name}")
    features, labels = [], []

    for label, cls in enumerate(["cats", "dogs"]):
        feat_folder = os.path.join(feat_dir, set_name, cls)
        feat_paths = sorted(glob(os.path.join(feat_folder, "*.npy")))
        print(f"  Found {len(feat_paths)} feature files in '{cls}'")

        for path in feat_paths:
            vec = load(path)  # Load 768-D feature vector directly
            features.append(vec)
            labels.append(label)

    features = array(features)
    labels = array(labels)
    print(f"  Feature shape: {features.shape}")

    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42, stratify=labels
    )

    # Logistic Regression
    print("Training Logistic Regression...")
    t0 = perf_counter()
    lr_model = LogisticRegression(max_iter=2000)
    lr_model.fit(X_train, y_train)
    t1 = perf_counter()
    lr_acc = accuracy_score(y_test, lr_model.predict(X_test))
    print(f"  Accuracy: {lr_acc:.4f} | Time: {t1 - t0:.2f}s")

    # SVM
    print("Training SVM (RBF kernel)...")
    t0 = perf_counter()
    svm_model = SVC(kernel="rbf", C=1.0, gamma="scale")
    svm_model.fit(X_train, y_train)
    t1 = perf_counter()
    svm_acc = accuracy_score(y_test, svm_model.predict(X_test))
    print(f"  Accuracy: {svm_acc:.4f} | Time: {t1 - t0:.2f}s")

    results[set_name] = {"LogReg": lr_acc, "SVM": svm_acc}

# --- Summary ---
print("\n==================== RESULTS SUMMARY ====================")
print(f"{'Dataset':<15} {'LogReg Acc':<15} {'SVM Acc':<15}")
print("----------------------------------------------------------")
for set_name, vals in results.items():
    print(f"{set_name:<15} {vals['LogReg']:<15.4f} {vals['SVM']:<15.4f}")
print("==========================================================")



Processing feature set: original
  Found 12290 feature files in 'cats'
  Found 12269 feature files in 'dogs'
  Found 12269 feature files in 'dogs'
  Feature shape: (24559, 768)
Training Logistic Regression...
  Feature shape: (24559, 768)
Training Logistic Regression...
  Accuracy: 0.9900 | Time: 0.30s
Training SVM (RBF kernel)...
  Accuracy: 0.9900 | Time: 0.30s
Training SVM (RBF kernel)...
  Accuracy: 0.9921 | Time: 5.68s

Processing feature set: enhanced
  Found 12490 feature files in 'cats'
  Accuracy: 0.9921 | Time: 5.68s

Processing feature set: enhanced
  Found 12490 feature files in 'cats'
  Found 12270 feature files in 'dogs'
  Found 12270 feature files in 'dogs'
  Feature shape: (24760, 768)
Training Logistic Regression...
  Feature shape: (24760, 768)
Training Logistic Regression...
  Accuracy: 0.9786 | Time: 0.82s
Training SVM (RBF kernel)...
  Accuracy: 0.9786 | Time: 0.82s
Training SVM (RBF kernel)...
  Accuracy: 0.9871 | Time: 9.92s

==================== RESULTS SUMMARY

### Task 1.6: Segmentation Consistency (SegFormer)

Checking if the enhanced images preserve object boundaries well enough for a Transformer-based segmentation model.

In [ ]:
print("="*80)
print("SEGMENTATION CONSISTENCY TEST (SegFormer - ViT Based)")
print("="*80)
print("Goal: Check if enhanced images preserve object boundaries recognizable by a Transformer-based model.")
print("Method: Compare SegFormer masks of Original vs Enhanced images (IoU).")
print("-" * 80)

import torch
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import numpy as np
from PIL import Image

# Load SegFormer (b0 is the smallest/fastest variant)
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
seg_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512").to('cpu')#trained on ADE20K dataset
seg_model.eval()

def get_segmentation_iou(image_path, model, processor):
    try:
        # Load Original
        img_orig = Image.open(image_path).convert("RGB")
        w, h = img_orig.size
        
        # Preprocess for SegFormer
        inputs = processor(images=img_orig, return_tensors="pt")
        
        # Get Original Mask
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits  # shape (batch_size, num_labels, height/4, width/4)
            
        # Upsample logits to original image size
        upsampled_logits = torch.nn.functional.interpolate(
            logits,
            size=img_orig.size[::-1], # (height, width)
            mode="bilinear",
            align_corners=False,
        )
        mask_orig = upsampled_logits.argmax(dim=1)[0].numpy()
        
        # Simulate Enhancement
        parts = image_path.split(os.sep)
        enh_path = os.path.join(base_path, "enhanced", parts[-2], parts[-1])
        
        if not os.path.exists(enh_path):
            return None
            
        img_enh = Image.open(enh_path).convert("RGB")
        
        # Preprocess Enhanced
        inputs_enh = processor(images=img_enh, return_tensors="pt")
        
        # Get Enhanced Mask
        with torch.no_grad():
            outputs_enh = model(**inputs_enh)
            logits_enh = outputs_enh.logits
            
        # Upsample Enhanced logits to ORIGINAL image size (crucial for IoU)
        upsampled_logits_enh = torch.nn.functional.interpolate(
            logits_enh,
            size=img_orig.size[::-1], # (height, width)
            mode="bilinear",
            align_corners=False,
        )
        mask_enh = upsampled_logits_enh.argmax(dim=1)[0].numpy()
            
        # Compute IoU
        intersection = np.logical_and(mask_orig, mask_enh).sum()
        union = np.logical_or(mask_orig, mask_enh).sum()
        
        if union == 0:
            return 1.0 
        
        return intersection / union

    except Exception as e:
        # print(f"Error: {e}")
        return None

iou_scores = []
for animal in ["cats", "dogs"]:
    folder = os.path.join(base_path, "original", animal)
    files = sorted([f for f in os.listdir(folder) if f.endswith(('.jpg', '.png'))])
    print(f"Found {len(files)} images for {animal}")
    
    for f in tqdm(files, desc=f"Segmentation Test ({animal})"):
        fpath = os.path.join(folder, f)
        iou = get_segmentation_iou(fpath, seg_model, processor)
        if iou is not None:
            iou_scores.append(iou)

avg_iou = np.mean(iou_scores) if iou_scores else 0
print(f"\nAverage IoU (Original vs Enhanced Masks): {avg_iou:.4f}")

if avg_iou > 0.75:
    print("HIGH CONSISTENCY: Enhanced images preserve object boundaries well.")
elif avg_iou > 0.6:
    print("MODERATE CONSISTENCY: Some boundary details are lost.")
else:
    print("LOW CONSISTENCY: Object shapes are significantly distorted.")
print("="*80)

SEGMENTATION CONSISTENCY TEST (SegFormer - ViT Based)
Goal: Check if enhanced images preserve object boundaries recognizable by a Transformer-based model.
Method: Compare SegFormer masks of Original vs Enhanced images (IoU).
--------------------------------------------------------------------------------
Found 12290 images for cats


Segmentation Test (cats): 100%|██████████| 12290/12290 [1:48:30<00:00,  1.89it/s]


Found 12269 images for dogs


Segmentation Test (dogs):  91%|█████████▏| 11209/12269 [1:39:50<08:57,  1.97it/s]/Users/priyanshusharma912o/miniconda3/envs/ML_project/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Segmentation Test (dogs): 100%|██████████| 12269/12269 [1:48:49<00:00,  1.88it/s]


Average IoU (Original vs Enhanced Masks): 0.9167
HIGH CONSISTENCY: Enhanced images preserve object boundaries well.
